# ಪಾಠ 04 - ಸಾಧನ ಬಳಕೆ ವಿನ್ಯಾಸ ಪ್ಯಾಟರ್ನ್

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೆ임್ವರ್ಕ್ (ಪೈಥಾನ್) ಬಳಸಿ AI ಏಜೆಂಟ್‌ಗಳಿಗೆ **ಸಾಧನ ಬಳಕೆ** ವಿನ್ಯಾಸ ಪ್ಯಾಟರ್ನ್ ಅನ್ನು ಕಲಿಯುತ್ತೀರಿ. ನಾವು ಒಳಗೊಂಡಿದ್ದು:

- `@tool` ಡೆಕೊರೇಟರ್ ಮತ್ತು ಟೈಪ್ಡ್ ಪ್ಯಾರಾಮೀಟರ್‌ಗಳೊಂದಿಗೆ ಫಂಕ್ಷನ್ ಸಾಧನಗಳನ್ನು ನಿರ್ಧರಿಸುವುದು
- ಮಾದರಿ ಪ್ರತಿಯೊಂದು ಸಾಧನ ಏನು ಮಾಡುತ್ತದೆ ಎಂದು ತಿಳಿದುಕೊಳ್ಳಲು ಸಾಧನ ವೃತ್ತರಚನೆಗಳನ್ನು ಒದಗಿಸುವುದು
- `approval_mode` ಮೂಲಕ ಸಾಧನ ಕಾರ್ಯಗತಗೊಳಿಸುವಿಕೆಯನ್ನು ನಿಯಂತ್ರಿಸುವುದು
- ಪೈಡ್ಯಾನ್ಟಿಕ್ ಮಾದರಿಗಳು ಮತ್ತು `response_format` ಮುಖಾಂತರ **ಸಂರಚಿತ output** ಅನ್ನು ಹಿಂತಿರುಗಿಸುವುದು

ಈ ದೃಶ್ಯಾಂತವು **ಪ್ರವಾಸ ಬುಕ್ಕಿಂಗ್ ಏಜೆಂಟ್** ಆಗಿದ್ದು, ಗಮ್ಯಸ್ಥಾನಗಳನ್ನು ನೋಡಬಹುದಾಗಿದೆ, ಲಭ್ಯತೆಯನ್ನು ಪರಿಶೀಲಿಸುತ್ತದೆ ಮತ್ತು ವಿಮಾನ ಮಾಹಿತಿ ಹಿಂತಿರುಗಿಸುವುದು.


## ವ್ಯವಸ್ಥೆ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool ಡೆಕೊರೇಟರ್‌ನೊಂದಿಗೆ ಸಾಧನಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು

`@tool` ಡೆಕೊರೇಟರ್ ಸರಳ Python ಫಂಕ್ಷನ್ ಅನ್ನು ಏಜೆಂಟ್ ಕರೆ ಮಾಡಬಹುದಾದ ಸಾಧನವಾಗಿಸುತ್ತದೆ.
ಮುಖ್ಯ ಅಂಶಗಳು:

- **ಡಾಕ್ಸ್ಟ್ರಿಂಗ್** ಮಾದರಿ ನೋಡಬಹುದಾದ ಸಾಧನ ವಿವರಣೆಯಾಗುತ್ತದೆ.
- **ಪ್ರಕಾರ ಅನೋಟೇಶನ್‌ಗಳು** (`Annotated` ವಿವರಣೆಗಳ ಜೊತೆಗೆ ಸೇರಿ) ಸಾಧನ ಸ್ಕೀಮಾ ನಿರ್ಧರಿಸುತ್ತವೆ.
- `approval_mode` ಬಳಕೆದಾರನಿಂದ ಪ್ರತಿ ಕರೆ ಅನುಮೋದನೆ ಬೇಕಾಗಿದೆಯೇ ಎಂದು ನಿಯಂತ್ರಿಸುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ಬಹು ಸಾಧನಗಳನ್ನು ಬಳಸಿ ಏಜೆಂಟ್ ರಚನೆ

ಮೂರೂ ಸಾಧನಗಳನ್ನು 모두 클ೈ언트에게 전달ಿಸಿರಿ, ಆದ್ದರಿಂದ ಮಾದರಿ ಬಳಕೆದಾರರ ಪ್ರಶ್ನೆಗೆ ಉತ್ತರಿಸಲು ಬೇಕಾದ ಯಾವುದೇ ಸಾಧನವನ್ನು ಕರೆಸಿಕೊಳ್ಳಬಹುದು.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ಟೂಲ್ಸ್‌ನೊಂದಿಗೆ ರಚಿಸಲ್ಪಟ್ಟ ಔಟ್‌ಪುಟ್

`response_format` ಅನ್ನು Pydantic ಮಾದರಿಗೆ ಹೊಂದಿಸುವ ಮೂಲಕ, ಏಜೆಂಟ್ ಸ್ವತಃ ಸ್ವಾತಂತ್ರ್ಯ ರೂಪದ ಪಠ್ಯದ ಬದಲಿಗೆ ಚೆನ್ನಾಗಿ ಟೈಪ್ ಮಾಡಿದ JSON ವಸ್ತುವನ್ನು ಹಿಂತಿರುಗಿಸಲು ಬಾಧ್ಯವಾಗುತ್ತದೆ. ಇದು ಕಳಿತುಬರೋಡಿ ಕೋಡ್ ಫಲಿತಾಂಶವನ್ನು ಪ್ರೋಗ್ರಾಮ್ಯಾಟ್ ರೀತಿಯಲ್ಲಿ ಬಳಸಬೇಕಾಗುವಾಗ ಉಪಯುಕ್ತವಾಗುತ್ತದೆ.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## সরঞ্জাম অনুমোদন প্যাটার্নগুলি

`@tool` এর `approval_mode` প্যারামিটার নিয়ন্ত্রণ করে কোনটি সরঞ্জাম কলগুলি সম্পাদনের আগে মানব অনুমোদন প্রয়োজন কি না:

| মোড | আচরণ |
|---|---|
| `"never_require"` | সরঞ্জাম স্বয়ংক্রিয়ভাবে চলে — কোনও ব্যবহারকারীর নিশ্চিতকরণ প্রয়োজন নেই। |
| `"always_require"` | প্রতিটি কলের আগে ব্যবহারকারীর অনুমোদন আবশ্যক। |

"always_require" ব্যবহার করুন এমন সরঞ্জামগুলির জন্য যেগুলির পার্শ্ব প্রতিক্রিয়া থাকে (যেমন ফ্লাইট বুকিং, ক্রেডিট কার্ড চার্জ করা) যাতে মানব অবস্থা বজায় থাকে।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಹೇಗೆ ಕಲಿತಿರಿ:

1. ಟೂಲ್ ಸ್ಕೀಮಾದಂತೆ ಟೈಪ್ ಮಾಡಲಾದ ಪ್ಯಾರಾಮೀಟರ್‌ಗಳು ಮತ್ತು ಡಾಕ್ಸ್ಟ್ರಿಂಗ್‌ಗಳೊಂದಿಗೆ `@tool` ಅಲಂಕಾರವನ್ನು ಬಳಸಿ **ಟೂಲ್‌ಗಳನ್ನು ನಿರupe ಡಿಸು**.
2. ಅಜೆಂಟ್ ಅವರಿಗೆ ಸಂಕೀರ್ಣ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸಲು ಕ್ರಮವಾಗಿ ಕರೆ ಮಾಡಲು **ಬಹು ಟೂಲ್‌ಗಳನ್ನು ಸಂಯೋಜಿಸುವುದು**.
3. `response_format` ಆಗಿ ಪೈಡಾಂಟಿಕ್ ಮಾದರಿಯನ್ನು ಹಿಂತಿರುಗಿಸುವ ಮೂಲಕ **ಸಂರಚಿತ ಆದೇಶವನ್ನು ಹಿಂತಿರುಗಿಸುವುದು**.
4. ಸಂವೇದನಾಶೀಲ ಕಾರ್ಯಗಳಿಗೆ ಮಾನವನು ಭಾಗವಾಗಿರಲು `approval_mode` ಬಳಸಿ **ಟೂಲ್ ಅನುಮೋದನೆಯನ್ನು ನಿಯಂತ್ರಿಸುವುದು**.

ಈ ಮಾದರಿಗಳು ವಿಶ್ವಾಸಾರ್ಹ, ಉತ್ಪಾದನಾ ಸಿದ್ಧ ಅಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ಮಿಸಲು, ಹೊರಗಿನ ವ್ಯವಸ್ಥೆಗಳೊಂದಿಗೆ ಸುರಕ್ಷಿತವಾಗಿ ಸಂವಹನ ಮಾಡಲು ಮೂಲಬಲವನ್ನು ರೂಪಿಸುತ್ತವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
